# 推理验证

SFT 训练完成，现在来验证模型是否真正学会了 Wordle 的格式规范。本节将启动推理服务器，用 vf-eval 进行标准化评测，对比 SFT 前后的效果。

---

## 推理服务器

`scripts/infer_server.py` 是一个极简的 OpenAI 兼容推理服务器，用 `transformers` + `torch_npu` 直接加载 SFT 权重，监听 8000 端口。

### 工作流程

![Inference Server Flow](./images/inference-server-flow.png)


### 启动服务器

```bash
pip install transformers
pip install torchvision==0.27.0+cpu --index-url https://download.pytorch.org/whl/cpu
python3 scripts/infer_server.py \
  --model ./outputs/checkpoint_wordle_sft/step-20 \
  --port 8000 &

# 验证服务器是否正常
curl http://localhost:8000/health
# → {"status": "ok"}
```

---

## vf-eval 评测

`vf-eval` 是 prime-rl 提供的命令行评测工具。它加载 Wordle 游戏环境，向推理服务器发送多轮对话请求，模拟完整游戏过程并计算 reward。

### 安装评测环境

```bash
pip install prime                                          # verifiers 框架
git clone https://github.com/PrimeIntellect-ai/prime-rl.git
cd prime-rl

git clone https://github.com/PrimeIntellect-ai/verifiers.git deps/verifiers
pip install deps/verifiers/environments/wordle              # Wordle 游戏环境

# 下载 NLTK 语料库（Wordle 游戏环境依赖）
python3 -c "
import nltk
nltk.download('words')
nltk.download('averaged_perceptron_tagger')
"
```

### 评测工作流程

![vf-eval Game Flow](./images/vf-eval-game-flow.png)


### 运行评测

```bash
vf-eval wordle \
  --provider openai \
  --model instruct-sft \
  --api-base-url http://localhost:8000/v1 \
  --num-examples 20 \
  --rollouts-per-example 3 \
  --max-concurrent 1 \
  --max-tokens 1024 \
  --temperature 0.6
```

| 参数 | 说明 |
|------|------|
| `--provider openai` | 使用 OpenAI 兼容 API |
| `--api-base-url` | 推理服务器地址 |
| `--num-examples 20` | 评测 20 局（每局不同秘密词） |
| `--rollouts-per-example 3` | 每局重复 3 次，取平均 |
| `--max-concurrent 1` | 串行请求（避免单线程服务器过载） |
| `--max-tokens 1024` | 每轮最多生成 1024 tokens |
| `--temperature 0.6` | 采样温度（>0 产生多样性） |

---

## SFT 前后对比

### 基础模型（基座模型 Qwen3-1.7B-Instruct，未 SFT）

| 指标 | 值 | 解读 |
|------|-----|------|
| `format_reward` | **0.200** | 几乎无法遵循 `<guess>...</guess>` 格式 |
| `avg reward` | **0.04** | 得分极低，主要来自偶然的格式匹配 |
| `correct_answer` | 0% | 基础模型不会玩 Wordle |
| `partial_answer` | 0% | 无有效的猜词行为 |
| `num_turns` | ~2.0 | 2 轮后即放弃（生成质量差，被环境判定无效） |
| 每轮推理时间 | 25-58s | 输出冗长的 rambling text |

### SFT 微调后

| 指标 | 值 | 解读 |
|------|-----|------|
| `format_reward` | **1.000** | ✅ 模型 100% 遵循 `<guess>word</guess>` 格式 — **SFT 目标达成** |
| `avg reward` | **0.40** | 主要来自 format (1.0×0.2) + partial（字母部分正确） |
| `correct_answer` | 0% | 20 步 SFT 不足以学会策略性猜词 — 需要 RL 阶段 |
| `partial_answer` | 0.20 | ~1 个字母 G/Y 正确，模型在试探但未收敛到正确策略 |
| `num_turns` | ~7.0 | 完整玩满 7 轮，稳定多轮对话 |
| 每轮推理时间 | ~2s | 输出简洁高效 |

---

## 结果解读

### SFT 做到了什么？

1. **格式对齐**：`format_reward` 从 0.2 → 1.0，模型 100% 按 `<guess>word</guess>` 格式输出。这是 SFT 最重要的贡献。
2. **稳定多轮对话**：`num_turns` 从 ~2 → ~7，模型能完整走完一局游戏而不中途崩溃。
3. **初步猜词能力**：`partial_answer` 从 0% 到有部分字母/位置正确，说明模型理解了 G/Y/X 反馈的含义并尝试利用。

### SFT 没做到什么？

1. **策略性猜词**：`correct_answer` 仍为 0%。模型不会系统地利用绿色字母固定位置、排除灰色字母。这是 RL 阶段的目标。
2. **高效的搜索策略**：`length_bonus` 为 0，说明模型无法在较少的轮数内猜出答案。

> **核心结论**：SFT 完成了"脚手架"的搭建——模型学会了对话格式和基本的反馈理解。策略性猜词能力交给 RL 阶段，让模型通过试错和奖励信号来优化。

---

## 评测指标详解

Wordle 环境的 reward 由 4 个组件构成：

| 指标 | 权重 | 计算方式 | 满分含义 |
|------|------|---------|--------|
| `correct_answer` | 1.0 | `guess == answer` 时 = 1.0 | 猜对秘密词 |
| `partial_answer` | 0.2/0.1 | `0.2 × num_greens + 0.1 × num_yellows` | 部分字母正确 |
| `length_bonus` | 1.0 | `correct / num_guesses` | 更少步数猜中奖励更高 |
| `format_reward` | 0.2 | XML 解析器校验 `<guess>...</guess>` | 格式合规 |

> **format_reward 权重最低但最重要**：如果格式不对（`format_reward`=0），游戏环境无法解析模型的猜测，后续的 `correct_answer` 和 `partial_answer` 都无从谈起。

---

## 小结

- `infer_server.py` 提供 OpenAI 兼容 API，加载 SFT 权重即可推理。
- `vf-eval` 模拟完整 Wordle 游戏过程，计算 4 维 reward。
- SFT 后 `format_reward` 从 0.2 → 1.0，模型 100% 遵循游戏格式。
- 策略性猜词（`correct_answer`）需要 RL 阶段训练。

下一章，我们将通过融合算子优化训练性能，在不改变模型行为的前提下提升吞吐量。

## 练习

1. （单选题）SFT 训练后，format_reward 从 0.2 提升到 1.0，而 correct_answer 仍为 0%。原因是什么？
    A. 模型完全没有被训练
    B. 格式学习是简单模式匹配（学 `<guess>` 标签即可），策略需要在复杂状态空间做最优决策，需要 RL 的试错机制
    C. 评测脚本有 bug
    D. 数据集太小

2. （单选题）Wordle 评测的 4 个 reward 组件中，哪个权重最低但最重要？
    A. correct_answer（1.0）
    B. partial_answer（0.1/0.2）
    C. length_bonus（1.0）
    D. format_reward（0.2）

3. （判断题）SFT 后模型 num_turns 从 ~2 提升到 ~7（稳定完成多轮对话），因为模型学会了按 `<guess>` 格式输出，并且理解了 G/Y/X 反馈的含义。


In [ ]:
!cat ./answer/03.04_answer.txt